In [ ]:
import pandas as pd
import numpy as npp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import shap
import joblib


print(f'Rebuilding data pipeline...')


AGG_RULES = {
    'OIL_RATE_NORM':         'mean',
    'AVG_WHP_P':             'mean',
    'AVG_DOWNHOLE_PRESSURE': 'mean',
    'WCT':                   'mean',
    'GOR':                   'median',
    'DRAWDOWN':              'mean',
    'DAYS_ON_PROD':          'max',
    'ON_STREAM_HRS':         'sum',
}

df = pd.read_parquet('../data/processed/cleaned.parquet')
df['DATEPRD'] = pd.to_datetime(df['DATEPRD'])
df = df.set_index('DATEPRD').sort_index()
df['OIL_RATE_NORM'] = df.groupby('NPD_WELL_BORE_NAME')['OIL_RATE_NORM'].transform(
    lambda x : x.clip(upper= x.quantile(0.99))
)

TARGET_WELLS = ['15/9-F-12', '15/9-F-14']
monthly_well_data = {}
for well_name,well_df in df.groupby('NPD_WELL_BORE_NAME'):
    if(well_name not in TARGET_WELLS):
        print(f'this well has no clean data')
        continue
    cols_availble = {k: v for k, v in AGG_RULES.items() if k in well_df.columns}
    monthly = well_df[list(cols_availble.keys())].resample('ME').agg(
        cols_availble
    ).dropna(subset=['OIL_RATE_NORM'])
    monthly_well_data[well_name] = monthly